In [ ]:
import numpy as np
import xarray as xr
import toml
import munch
from tqdm import tqdm
import torch
import datetime

import warnings
warnings.filterwarnings("ignore")

from Fires._utilities.utils_mlflow import load_model_from_mlflow
from Fires._utilities.utils_inference import get_cmip6_inference

In [ ]:
config = munch.munchify(toml.load("/ceph/hpc/home/ciangottinid/ML4Fires/config/cmip6_inference.toml"))
searfire_ds_path = "/ceph/hpc/home/ciangottinid/ML4Fires_data/data_100km.zarr"
seafire_ds = xr.open_zarr(searfire_ds_path)

In [ ]:
import ipywidgets as widgets

scenario = widgets.Dropdown(
    options=[('SSP126', 'ssp126'), ('SSP245', 'ssp245'), ('SSP370', 'ssp370'),
             ('SSP585', 'ssp585')],
    value = 'ssp126',
    style={'description_width': '60px'}, 
    description='Scenario', disabled=False,
    layout=widgets.Layout(width='200px'))

climate_model = widgets.Dropdown(
    options=[('MPI-ESM1-2-HR', 'MPI-ESM1-2-HR'), ('CMCC-ESM2', 'CMCC-ESM2'), ('NorESM2-MM', 'NorESM2-MM'),
             ('CESM2', 'CESM2')],
    value = 'CMCC-ESM2',
    style={'description_width': '60px'}, 
    description='Model', disabled=False,
    layout=widgets.Layout(width='200px'))

year_range = widgets.IntRangeSlider(
    value=[2030, 2035],        # initial range
    min=2015,                 # min value
    max=2100,               # max value
    step=1,                # step size
    description='Year range:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='400px')
)

display(widgets.HBox([scenario, climate_model, year_range]))

In [ ]:
assert scenario.value != None, "Please select a CMIP6 scenario in the previous cell before proceesind"

In [ ]:
# run_name=input()
run_name="last"
registered_model = load_model_from_mlflow(run_name, provenance=True)
# registered_model

In [ ]:
predictions = get_cmip6_inference(
    seafire_ds=seafire_ds,
    run_name=run_name,
    scenario=scenario,
    climate_model=climate_model,
    year_range=year_range,
    infer_config=config,
    model=registered_model)


In [ ]:
monthly_agg = widgets.Dropdown(
    options=[('Mean', 'mean'), ('Sum', 'sum'), ('Median', 'median'),
             ('Mode', 'mode')],
    value = 'sum',
    style={'description_width': '150px'}, 
    description='Monthly aggregate', disabled=False,)

monthly_add_period = widgets.Dropdown(
    options=[('1 Month', '1M'), ('2 Months', '2M'), ('4 Months', '4M'),
             ('6 Month', '6M')],
    value = '1M',
    style={'description_width': '150px'}, 
    description='Monthly aggregate period', disabled=False,)

yearly_agg = widgets.Dropdown(
    options=[('Mean', 'mean'), ('Sum', 'sum'), ('Median', 'median'),
             ('Mode', 'mode')],
    value = 'mean',
    style={'description_width': '150px'}, 
    description='Yearly aggregate', disabled=False,)

decadal_agg = widgets.Dropdown(
    options=[('Mean', 'mean'), ('Sum', 'sum'), ('Median', 'median'),
             ('Mode', 'mode')],
    value = 'mean',
    style={'description_width': '150px'}, 
    description='Decadal aggregate', disabled=False,)

display(widgets.HBox([monthly_agg,
                      monthly_add_period]))
        
display(widgets.VBox([yearly_agg,
                      decadal_agg]))


In [ ]:
from Fires._utilities.utils_inference import process_and_plot_cmip6infer, process_and_plot_data, load_input_data
input_data = load_input_data(searfire_ds_path, '2019', '2020') # Required by internal setting of variables in Fires._utilities.utils_inference

temporal_aggregate_scheme = {"monthly":[monthly_agg.value,monthly_add_period.value],
                             "yearly":yearly_agg.value,
                            "decadal":decadal_agg.value}
process_and_plot_cmip6infer(
	data=predictions["global_burned_areas"],
	temporal_aggregate_scheme=temporal_aggregate_scheme,
	label=f'Averaged burned area for {year_range.value[0]}-{year_range.value[1]} in scenario {scenario.value}',
	lats=seafire_ds.latitude.values,
	lons=seafire_ds.longitude.values,
    scale_min=0,
    scale_max=8000,
	model_name="Unet ++"
)